In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/formula-1-race-result-classification-challenge/sample_submission.csv
/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv
/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv


In [2]:
import pandas as pd

# Correct dataset path
train = pd.read_csv(
    '/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

# Show info
print("Shape:", train.shape)

print("\nColumns:")
print(train.columns)

train.head()

Shape: (25749, 28)

Columns:
Index(['id', 'raceId', 'driverId', 'constructorId', 'year', 'round',
       'circuitId', 'lat', 'lng', 'alt', 'track_length_km', 'number_of_turns',
       'grid', 'driver_age', 'driver_nationality_code',
       'constructor_nationality_code', 'q1_ms', 'q2_ms', 'q3_ms',
       'best_qual_ms', 'pit_stop_count', 'avg_pit_ms', 'rolling_avg_position',
       'career_wins_so_far', 'prev_championship_points',
       'prev_championship_position', 'prev_constructor_points',
       'finishing_position'],
      dtype='object')


,id,raceId,driverId,constructorId,year,round,circuitId,lat,lng,alt,...,q3_ms,best_qual_ms,pit_stop_count,avg_pit_ms,rolling_avg_position,career_wins_so_far,prev_championship_points,prev_championship_position,prev_constructor_points,finishing_position
0,833_579,833,579,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,12
1,833_642,833,642,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
2,833_686,833,686,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3
3,833_786,833,786,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,2
4,833_589,833,589,105,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,18


In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Load datasets
train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

sample = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/sample_submission.csv'
)

# Select important beginner features
features = [
    'grid',
    'rolling_avg_position',
    'track_length_km',
    'number_of_turns'
]

# Training data
X_train = train[features]
y_train = train['finishing_position']

# Train model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Predict
X_test = test[features]
predictions = model.predict(X_test)

# Round to integers
predictions = predictions.round().astype(int)

# Create submission file
submission = sample.copy()
submission['finishing_position'] = predictions

# Save file
submission.to_csv('submission.csv', index=False)

print("Submission file created!")

Submission file created!


In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

sample = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/sample_submission.csv'
)

features = [
    'grid',
    'rolling_avg_position',
    'track_length_km',
    'number_of_turns',
    'driver_age',
    'career_wins_so_far',
    'prev_championship_points',
    'prev_constructor_points'
]

train[features] = train[features].fillna(0)
test[features] = test[features].fillna(0)

X_train = train[features]
y_train = train['finishing_position']

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(test[features])

predictions = predictions.round().astype(int)

submission = sample.copy()
submission['finishing_position'] = predictions

submission.to_csv('submission_v2.csv', index=False)

print("Improved submission file created!")

Improved submission file created!


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Load Data (FIXED PATHS)

train_df = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test_df = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

sample_df = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/sample_submission.csv'
)

print(f"Initial Shapes: Train {train_df.shape}, Test {test_df.shape}")

def engineer_features(df, is_train=True):

    df = df.copy()

    df = df.sort_values(['year', 'round', 'raceId'])

    df['best_qual_ms'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x.fillna(x.median() if not x.isna().all() else 0)
    )

    df['qual_gap'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_delta'] = df['grid'] - 1

    df['grid_norm'] = df.groupby('raceId')['grid'].transform(
        lambda x: x / x.max() if x.max() > 0 else 0
    )

    df['qual_rank'] = df.groupby('raceId')['best_qual_ms'].rank(method='first')

    df['grid_rank'] = df.groupby('raceId')['grid'].rank(method='first')

    df['qual_rel_strength'] = df['qual_rank'] / df.groupby(
        'raceId')['qual_rank'].transform('max')

    df['grid_rel_strength'] = df['grid_rank'] / df.groupby(
        'raceId')['grid_rank'].transform('max')

    df['rolling_trend'] = df.groupby('driverId')[
        'rolling_avg_position'
    ].diff().fillna(0)

    if is_train:

        c_perf = df.groupby(
            ['raceId', 'constructorId']
        )['finishing_position'].mean().reset_index()

        c_perf.columns = [
            'raceId',
            'constructorId',
            'c_race_avg'
        ]

        df = df.merge(
            c_perf,
            on=['raceId', 'constructorId'],
            how='left'
        )

        df['constructor_form'] = df.groupby(
            'constructorId'
        )['c_race_avg'].transform(
            lambda x: x.shift().rolling(window=5, min_periods=1).mean()
        )

        df = df.drop(columns=['c_race_avg'])

    else:

        df['constructor_form'] = df.groupby(
            'constructorId'
        )['rolling_avg_position'].transform(
            lambda x: x.rolling(window=5, min_periods=1).mean()
        )

    df['grid_qual_inter'] = df['grid'] * df['qual_gap']

    return df


train_eng = engineer_features(train_df, True)
test_eng = engineer_features(test_df, False)

# Fill missing values

num_cols = train_eng.select_dtypes(include=[np.number]).columns.tolist()

if 'finishing_position' in num_cols:
    num_cols.remove('finishing_position')

for col in num_cols:

    med = train_eng[col].median()

    train_eng[col] = train_eng[col].fillna(med)
    test_eng[col] = test_eng[col].fillna(med)


train_set = train_eng[train_eng['year'] <= 2018].copy()
valid_set = train_eng[train_eng['year'] > 2018].copy()

drop_feats = ['id', 'raceId', 'finishing_position', 'year', 'round']

features = [c for c in train_eng.columns if c not in drop_feats]

X_train = train_set[features]
y_train = train_set['finishing_position']

X_valid = valid_set[features]
y_valid = valid_set['finishing_position']

X_test = test_eng[features]

model = HistGradientBoostingRegressor(
    loss='absolute_error',
    learning_rate=0.03,
    max_iter=3000,
    max_leaf_nodes=127,
    min_samples_leaf=20,
    random_state=42
)

model.fit(X_train, y_train)

def apply_positional_ranking(df, raw_preds):

    temp = df[['raceId']].copy()

    temp['preds'] = np.clip(raw_preds, 1, 20)

    temp = temp.sort_values(['raceId', 'preds'])

    temp['finishing_position'] = temp.groupby(
        'raceId'
    ).cumcount() + 1

    return temp['finishing_position']


raw_val_preds = model.predict(X_valid)

valid_set['ranked_pos'] = apply_positional_ranking(
    valid_set,
    raw_val_preds
)

mae_after = mean_absolute_error(
    y_valid,
    valid_set['ranked_pos']
)

print("Validation MAE:", mae_after)

test_raw_preds = model.predict(X_test)

test_eng['finishing_position'] = apply_positional_ranking(
    test_eng,
    test_raw_preds
)

submission = sample_df.copy()

submission['finishing_position'] = test_eng[
    'finishing_position'
]

submission.to_csv('submission_hgb.csv', index=False)

print("submission_hgb.csv created!")

Initial Shapes: Train (25749, 28), Test (919, 27)
Validation MAE: 3.395121951219512
submission_hgb.csv created!


In [6]:
import pandas as pd

train_df = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

print(train_df.columns)

Index(['id', 'raceId', 'driverId', 'constructorId', 'year', 'round',
       'circuitId', 'lat', 'lng', 'alt', 'track_length_km', 'number_of_turns',
       'grid', 'driver_age', 'driver_nationality_code',
       'constructor_nationality_code', 'q1_ms', 'q2_ms', 'q3_ms',
       'best_qual_ms', 'pit_stop_count', 'avg_pit_ms', 'rolling_avg_position',
       'career_wins_so_far', 'prev_championship_points',
       'prev_championship_position', 'prev_constructor_points',
       'finishing_position'],
      dtype='object')


In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# Load Data

train_df = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test_df = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

sample_df = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/sample_submission.csv'
)

print("Loaded data")

# Feature Engineering

def create_features(df):

    df = df.copy()

    df = df.sort_values(['year','round','driverId'])

    # Qualifying gap
    df['best_qual_ms'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x.fillna(x.median())
    )

    df['qual_gap'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    # Grid features
    df['grid_delta'] = df['grid'] - 1

    df['grid_norm'] = df.groupby('raceId')['grid'].transform(
        lambda x: x / x.max()
    )

    # Ranking features
    df['qual_rank'] = df.groupby('raceId')['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby('raceId')['grid'].rank()

    # Rolling trend
    df['rolling_trend'] = df.groupby('driverId')[
        'rolling_avg_position'
    ].diff().fillna(0)

    # Interaction feature
    df['grid_qual_inter'] = df['grid'] * df['qual_gap']

    df = df.fillna(-1)

    return df


train_eng = create_features(train_df)
test_eng = create_features(test_df)

# Split train/valid

train_set = train_eng[train_eng['year'] <= 2018]
valid_set = train_eng[train_eng['year'] > 2018]

drop_cols = ['id','raceId','finishing_position','year']

features = [c for c in train_eng.columns if c not in drop_cols]

X_train = train_set[features]
y_train = train_set['finishing_position']

X_valid = valid_set[features]
y_valid = valid_set['finishing_position']

X_test = test_eng[features]

# LightGBM Model

model = lgb.LGBMRegressor(
    objective='regression_l1',
    learning_rate=0.03,
    num_leaves=127,
    n_estimators=4000,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric='mae',
    callbacks=[lgb.early_stopping(200)]
)

print("Model trained")

# Ranking Function

def rank_positions(df, preds):

    temp = df[['raceId']].copy()

    temp['pred'] = preds

    temp['finishing_position'] = temp.groupby(
        'raceId'
    )['pred'].rank(method='first').astype(int)

    return temp['finishing_position']


# Validation check

val_preds = model.predict(X_valid)

valid_set['ranked'] = rank_positions(
    valid_set,
    val_preds
)

val_mae = mean_absolute_error(
    y_valid,
    valid_set['ranked']
)

print("Validation MAE:", val_mae)

# Test prediction

test_preds = model.predict(X_test)

test_eng['finishing_position'] = rank_positions(
    test_eng,
    test_preds
)

submission = sample_df.copy()

submission['finishing_position'] = test_eng[
    'finishing_position'
]

submission.to_csv('submission_lgb_strong.csv', index=False)

print("submission_lgb_strong.csv created!")

Loaded data
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003765 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4349
[LightGBM] [Info] Number of data points in the train set: 24109, number of used features: 31
[LightGBM] [Info] Start training from score 13.000000
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[291]	valid_0's l1: 3.29998
Model trained
Validation MAE: 3.40609756097561
submission_lgb_strong.csv created!


/tmp/ipykernel_16/612000482.py:125: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_set['ranked'] = rank_positions(


In [8]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/formula-1-race-result-classification-challenge/sample_submission.csv
/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv
/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv


In [9]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# =========================
# LOAD DATA (CORRECT PATH)
# =========================

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

# =========================
# FEATURE ENGINEERING
# =========================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['year','round','driverId'])

    # Fix missing qualifying time
    df['best_qual_ms'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    # Qualification gap
    df['qual_gap'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    # Grid normalized
    df['grid_norm'] = df.groupby('raceId')['grid'].transform(
        lambda x: x / x.max()
    )

    # Grid delta
    df['grid_delta'] = df['grid'] - 1

    # Rolling trend
    df['rolling_trend'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].diff().fillna(0)

    # Relative ranks
    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    df = df.fillna(-1)

    # Categorical features
    cat_cols = [
        'driverId',
        'constructorId',
        'circuitId',
        'driver_nationality_code',
        'constructor_nationality_code'
    ]

    for col in cat_cols:
        df[col] = df[col].astype('category')

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# =========================
# TIME SPLIT
# =========================

train_set = train[train['year'] <= 2018].copy()
valid_set = train[train['year'] > 2018].copy()

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set['finishing_position']

X_valid = valid_set[features]
y_valid = valid_set['finishing_position']

# =========================
# MODEL
# =========================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.03,

    num_leaves=63,

    feature_fraction=0.9,

    bagging_fraction=0.8,

    bagging_freq=5,

    n_estimators=4000,

    random_state=42,

    n_jobs=-1
)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(200),
        lgb.log_evaluation(100)

    ]
)

# =========================
# VALIDATION
# =========================

val_preds = model.predict(X_valid)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# =========================
# TEST PREDICTION (FINAL FIXED)
# =========================

# Raw predictions (NO rounding)
test['pred_temp'] = model.predict(
    test[features]
)

# Rank within each race
test['finishing_position'] = test.groupby(
    'raceId'
)['pred_temp'].rank(
    method='first'
).astype(int)

# Ensure valid positions
test['finishing_position'] = np.clip(
    test['finishing_position'],
    1,
    20
)

# Create submission
submission = test[
    ['raceId','driverId','finishing_position']
]

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead 

In [10]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# =========================
# LOAD DATA (CORRECT PATH)
# =========================

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

# =========================
# FEATURE ENGINEERING
# =========================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['year','round','driverId'])

    # Fix missing qualifying time
    df['best_qual_ms'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    # Qualification gap
    df['qual_gap'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    # Grid normalized
    df['grid_norm'] = df.groupby('raceId')['grid'].transform(
        lambda x: x / x.max()
    )

    # Grid delta
    df['grid_delta'] = df['grid'] - 1

    # Rolling trend
    df['rolling_trend'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].diff().fillna(0)

    # Relative ranks
    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    df = df.fillna(-1)

    # Categorical features
    cat_cols = [
        'driverId',
        'constructorId',
        'circuitId',
        'driver_nationality_code',
        'constructor_nationality_code'
    ]

    for col in cat_cols:
        df[col] = df[col].astype('category')

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# =========================
# TIME SPLIT
# =========================

train_set = train[train['year'] <= 2018].copy()
valid_set = train[train['year'] > 2018].copy()

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set['finishing_position']

X_valid = valid_set[features]
y_valid = valid_set['finishing_position']

# =========================
# MODEL
# =========================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.03,

    num_leaves=63,

    feature_fraction=0.9,

    bagging_fraction=0.8,

    bagging_freq=5,

    n_estimators=4000,

    random_state=42,

    n_jobs=-1
)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(200),
        lgb.log_evaluation(100)

    ]
)

# =========================
# VALIDATION
# =========================

val_preds = model.predict(X_valid)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# =========================
# TEST PREDICTION (FINAL FIX)
# =========================

# Raw predictions
test['pred_temp'] = model.predict(
    test[features]
)

# Rank inside each race
test['finishing_position'] = test.groupby(
    'raceId'
)['pred_temp'].rank(
    method='first'
).astype(int)

# Clip valid range
test['finishing_position'] = np.clip(
    test['finishing_position'],
    1,
    20
)

# =========================
# FINAL SUBMISSION FORMAT (CORRECT)
# =========================

submission = pd.DataFrame()

submission['id'] = test['id']

submission['finishing_position'] = test['finishing_position']

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead 

In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# =========================
# LOAD DATA (CORRECT PATH)
# =========================

train = pd.read_csv(
    '/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
    '/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

# =========================
# FEATURE ENGINEERING
# =========================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['year','round','driverId'])

    # -------------------------
    # QUALIFYING FEATURES
    # -------------------------

    df['best_qual_ms'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    # -------------------------
    # GRID FEATURES
    # -------------------------

    df['grid_norm'] = df.groupby('raceId')['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # -------------------------
    # DRIVER MOMENTUM
    # -------------------------

    df['rolling_trend'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].diff().fillna(0)

    df['driver_recent_form'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['driver_long_form'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    # -------------------------
    # CONSTRUCTOR FEATURES
    # -------------------------

    df['constructor_recent_form'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['constructor_long_form'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    # -------------------------
    # CHAMPIONSHIP FEATURES
    # -------------------------

    df['champ_points_total'] = (
        df['prev_championship_points'] +
        df['prev_constructor_points']
    )

    df['career_strength'] = df.groupby(
        'driverId'
    )['career_wins_so_far'].transform(
        'max'
    )

    # -------------------------
    # CLEAN MISSING
    # -------------------------

    df = df.fillna(-1)

    # -------------------------
    # CATEGORICAL FEATURES
    # -------------------------

    cat_cols = [

        'driverId',
        'constructorId',
        'circuitId',
        'driver_nationality_code',
        'constructor_nationality_code'

    ]

    for col in cat_cols:
        df[col] = df[col].astype('category')

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# =========================
# TIME SPLIT
# =========================

train_set = train[train['year'] <= 2018].copy()
valid_set = train[train['year'] > 2018].copy()

drop_cols = [

    'id',
    'raceId',
    'finishing_position',
    'year'

]

features = [

    c for c in train.columns
    if c not in drop_cols

]

X_train = train_set[features]
y_train = train_set['finishing_position']

X_valid = valid_set[features]
y_valid = valid_set['finishing_position']

# =========================
# MODEL
# =========================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.025,

    num_leaves=63,

    feature_fraction=0.9,

    bagging_fraction=0.8,

    bagging_freq=5,

    n_estimators=5000,

    random_state=42,

    n_jobs=-1

)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(300),
        lgb.log_evaluation(200)

    ]

)

# =========================
# VALIDATION CHECK
# =========================

val_preds = model.predict(X_valid)

val_preds = np.clip(

    np.round(val_preds),
    1,
    20

)

val_mae = mean_absolute_error(

    y_valid,
    val_preds

)

print("Validation MAE:", val_mae)

# =========================
# TEST PREDICTION
# =========================

test_preds = model.predict(

    test[features]

)

test_preds = np.clip(

    np.round(test_preds),
    1,
    20

)

test['finishing_position'] = test_preds

# -------------------------
# RACE-WISE RANK FIX
# -------------------------

test['finishing_position'] = test.groupby(

    'raceId'

)['finishing_position'].rank(

    method='first'

).astype(int)

# =========================
# FINAL SUBMISSION FORMAT
# =========================

submission = pd.DataFrame()

submission['id'] = test['id']

submission['finishing_position'] = test['finishing_position']

submission.to_csv(

    'submission.csv',
    index=False

)

print("submission.csv created successfully")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead 

In [12]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# =========================
# LOAD DATA
# =========================

train = pd.read_csv(
    '/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
    '/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

# =========================
# FEATURE ENGINEERING
# =========================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['year','round','driverId'])

    # QUALIFY FEATURES

    df['best_qual_ms'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    # GRID FEATURES

    df['grid_norm'] = df.groupby('raceId')['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # DRIVER MOMENTUM

    df['rolling_trend'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].diff().fillna(0)

    df['driver_recent_form'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['driver_long_form'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    # CONSTRUCTOR FORM

    df['constructor_recent_form'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['constructor_long_form'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    # CHAMPIONSHIP FEATURES

    df['champ_points_total'] = (
        df['prev_championship_points'] +
        df['prev_constructor_points']
    )

    df['career_strength'] = df.groupby(
        'driverId'
    )['career_wins_so_far'].transform(
        'max'
    )

    df = df.fillna(-1)

    cat_cols = [

        'driverId',
        'constructorId',
        'circuitId',
        'driver_nationality_code',
        'constructor_nationality_code'

    ]

    for col in cat_cols:
        df[col] = df[col].astype('category')

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# =========================
# TARGET ENCODING (HIGH IMPACT)
# =========================

def add_target_encoding(train_df, test_df):

    cols = [

        'driverId',
        'constructorId',
        'circuitId'

    ]

    for col in cols:

        target_mean = train_df.groupby(col)[
            'finishing_position'
        ].mean()

        train_df[col + '_te'] = train_df[col].map(target_mean)

        test_df[col + '_te'] = test_df[col].map(target_mean)

        test_df[col + '_te'] = test_df[
            col + '_te'
        ].fillna(
            target_mean.mean()
        )

    return train_df, test_df


train, test = add_target_encoding(train, test)

# =========================
# TIME SPLIT
# =========================

train_set = train[train['year'] <= 2018].copy()
valid_set = train[train['year'] > 2018].copy()

drop_cols = [

    'id',
    'raceId',
    'finishing_position',
    'year'

]

features = [

    c for c in train.columns
    if c not in drop_cols

]

X_train = train_set[features]
y_train = train_set['finishing_position']

X_valid = valid_set[features]
y_valid = valid_set['finishing_position']

# =========================
# MODEL
# =========================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.02,

    num_leaves=63,

    feature_fraction=0.9,

    bagging_fraction=0.8,

    bagging_freq=5,

    n_estimators=6000,

    random_state=42,

    n_jobs=-1

)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(400),
        lgb.log_evaluation(200)

    ]

)

# =========================
# VALIDATION
# =========================

val_preds = model.predict(X_valid)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# =========================
# TEST PREDICTION
# =========================

test_preds = model.predict(
    test[features]
)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

# =========================
# RACE RANKING FIX
# =========================

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# =========================
# FINAL SUBMISSION
# =========================

submission = pd.DataFrame()

submission['id'] = test['id']

submission['finishing_position'] = test['finishing_position']

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

/tmp/ipykernel_16/2866431386.py:136: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = train_df.groupby(col)[
/tmp/ipykernel_16/2866431386.py:136: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = train_df.groupby(col)[
/tmp/ipykernel_16/2866431386.py:136: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = train_df.groupby(col)[


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead 

In [13]:
import pandas as pd
import numpy as np

import lightgbm as lgb

from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor

# =========================
# LOAD DATA
# =========================

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

# =========================
# FEATURE ENGINEERING
# =========================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['year','round','driverId'])

    # QUALIFY

    df['best_qual_ms'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    # GRID

    df['grid_norm'] = df.groupby('raceId')['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # DRIVER MOMENTUM

    df['rolling_trend'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].diff().fillna(0)

    df['driver_recent_form'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['driver_long_form'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    # CONSTRUCTOR

    df['constructor_recent_form'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['constructor_long_form'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    # CHAMPIONSHIP

    df['champ_points_total'] = (

        df['prev_championship_points'] +
        df['prev_constructor_points']

    )

    df['career_strength'] = df.groupby(
        'driverId'
    )['career_wins_so_far'].transform(
        'max'
    )

    df = df.fillna(-1)

    cat_cols = [

        'driverId',
        'constructorId',
        'circuitId',
        'driver_nationality_code',
        'constructor_nationality_code'

    ]

    for col in cat_cols:

        df[col] = df[col].astype('category')

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# =========================
# TARGET ENCODING
# =========================

def add_target_encoding(train_df, test_df):

    cols = [

        'driverId',
        'constructorId',
        'circuitId'

    ]

    for col in cols:

        target_mean = train_df.groupby(
            col,
            observed=False
        )['finishing_position'].mean()

        train_df[col + '_te'] = train_df[col].map(target_mean)

        test_df[col + '_te'] = test_df[col].map(target_mean)

        test_df[col + '_te'] = test_df[
            col + '_te'
        ].fillna(
            target_mean.mean()
        )

    return train_df, test_df


train, test = add_target_encoding(train, test)

# =========================
# SPLIT
# =========================

train_set = train[train['year'] <= 2018].copy()
valid_set = train[train['year'] > 2018].copy()

drop_cols = [

    'id',
    'raceId',
    'finishing_position',
    'year'

]

features = [

    c for c in train.columns
    if c not in drop_cols

]

X_train = train_set[features]
y_train = train_set['finishing_position']

X_valid = valid_set[features]
y_valid = valid_set['finishing_position']

# =========================
# LIGHTGBM MODEL
# =========================

lgb_model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.02,

    num_leaves=63,

    feature_fraction=0.9,

    bagging_fraction=0.8,

    bagging_freq=5,

    n_estimators=6000,

    random_state=42,

    n_jobs=-1

)

lgb_model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(400),
        lgb.log_evaluation(200)

    ]

)

# =========================
# EXTRA TREES MODEL
# =========================

et_model = ExtraTreesRegressor(

    n_estimators=400,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

et_model.fit(

    X_train,
    y_train

)

# =========================
# VALIDATION CHECK
# =========================

lgb_val = lgb_model.predict(X_valid)

et_val = et_model.predict(X_valid)

val_preds = (

    0.7 * lgb_val +
    0.3 * et_val

)

val_preds = np.clip(

    np.round(val_preds),
    1,
    20

)

val_mae = mean_absolute_error(

    y_valid,
    val_preds

)

print("Validation MAE:", val_mae)

# =========================
# TEST PREDICTIONS
# =========================

lgb_test = lgb_model.predict(test[features])

et_test = et_model.predict(test[features])

final_preds = (

    0.7 * lgb_test +
    0.3 * et_test

)

final_preds = np.clip(

    np.round(final_preds),
    1,
    20

)

test['finishing_position'] = final_preds

# =========================
# RACE RANK FIX
# =========================

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# =========================
# SUBMISSION
# =========================

submission = pd.DataFrame()

submission['id'] = test['id']

submission['finishing_position'] = test['finishing_position']

submission.to_csv(

    'submission.csv',
    index=False

)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead 

In [14]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error

# =========================
# LOAD DATA
# =========================

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

# =========================
# FEATURE ENGINEERING
# =========================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['year','round','driverId'])

    # QUALIFY FEATURES

    df['best_qual_ms'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby('raceId')['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    # GRID FEATURES

    df['grid_norm'] = df.groupby('raceId')['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # DRIVER FORM

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['driver_form_10'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    # CONSTRUCTOR FORM

    df['constructor_form_5'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['constructor_form_10'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# =========================
# TRAIN SPLIT
# =========================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set['finishing_position']

X_valid = valid_set[features]
y_valid = valid_set['finishing_position']

# =========================
# MODEL 1 — LIGHTGBM
# =========================

lgb_model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.02,

    num_leaves=63,

    n_estimators=7000,

    random_state=42,

    n_jobs=-1

)

lgb_model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(400),
        lgb.log_evaluation(200)

    ]
)

# =========================
# MODEL 2 — EXTRA TREES
# =========================

et_model = ExtraTreesRegressor(

    n_estimators=400,

    max_depth=14,

    random_state=42,

    n_jobs=-1

)

et_model.fit(
    X_train,
    y_train
)

# =========================
# VALIDATION CHECK
# =========================

lgb_val = lgb_model.predict(X_valid)
et_val = et_model.predict(X_valid)

val_preds = (

    0.75*lgb_val +
    0.25*et_val

)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# =========================
# TEST PREDICTIONS
# =========================

lgb_test = lgb_model.predict(test[features])
et_test = et_model.predict(test[features])

final_preds = (

    0.75*lgb_test +
    0.25*et_test

)

test['raw_pred'] = final_preds

# =========================
# SAFE RACE RANKING
# =========================

test = test.sort_values(
    ['raceId', 'raw_pred']
)

test['finishing_position'] = (

    test.groupby('raceId')
    .cumcount()
    + 1

)

test.drop(
    columns=['raw_pred'],
    inplace=True
)

# =========================
# SUBMISSION
# =========================

submission = pd.DataFrame()

submission['id'] = test['id']

submission['finishing_position'] = test[
    'finishing_position'
]

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001679 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4845
[LightGBM] [Info] Number of data points in the train set: 24109, number of used features: 32
[LightGBM] [Info] Start training from score 13.000000
Training until validation scores don't improve for 400 rounds
[200]	valid_0's l1: 3.26438
[400]	valid_0's l1: 3.26645
[600]	valid_0's l1: 3.28397
Early stopping, best iteration is:
[270]	valid_0's l1: 3.25571
Validation MAE: 3.265853658536585
submission.csv created


In [15]:
# ============================================================
# STABLE COMPETITIVE LIGHTGBM (SAFE UPGRADE)
# No broken CV
# Strong features
# Reliable leaderboard improvement
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# ============================================================
# LOAD DATA
# ============================================================

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['year','round','driverId'])

    # Fill missing qualifying
    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    # Gap features
    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    # Trend
    df['rolling_trend'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].diff().fillna(0)

    # Rank features
    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # NEW STRONG FEATURES
    df['driver_mean_pos'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform('mean')

    df['constructor_mean_pos'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform('mean')

    df['driver_std_pos'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform('std')

    df['constructor_std_pos'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform('std')

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# TIME SPLIT (SAFE)
# ============================================================

train_set = train[train['year'] <= 2018].copy()
valid_set = train[train['year'] > 2018].copy()

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

# ============================================================
# MODEL
# ============================================================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.02,

    num_leaves=63,

    max_depth=-1,

    feature_fraction=0.85,

    bagging_fraction=0.85,

    bagging_freq=5,

    n_estimators=6000,

    random_state=42,

    n_jobs=-1
)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(400),
        lgb.log_evaluation(200)

    ]
)

# ============================================================
# VALIDATION
# ============================================================

val_preds = model.predict(X_valid)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# ============================================================
# TEST
# ============================================================

test_preds = model.predict(
    test[features]
)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

# ============================================================
# RACE RANK FIX (SAFE)
# ============================================================

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# ============================================================
# SUBMISSION
# ============================================================

submission = pd.DataFrame({

    'id': test_ids,
    'finishing_position': test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.85, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.85
[LightGBM] [Warning] bagging_fraction is set=0.85, subsample=1.0 will be ignored. Current value: bagging_fraction=0.85
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.85, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.85
[LightGBM] [Warning] bagging_fraction is set=0.85, subsample=1.0 will be ignored. Current value: bagging_fraction=0.85
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001652 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4900
[LightGBM] [Info] Number of data p

In [16]:
# ============================================================
# PERFORMANCE FEATURE UPGRADE VERSION
# Based on your stable Version 10
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# LOAD DATA

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['driverId','year','round'])

    # Fill qualifying
    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    # Qual gap
    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    # Grid normalized
    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    # Rank features
    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # =====================
    # NEW STRONG FEATURES
    # =====================

    # Driver long-term strength
    df['driver_mean_pos'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform('mean')

    # Driver recent form
    df['driver_recent_form'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5,min_periods=1).mean()
    )

    # Constructor strength
    df['constructor_strength'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform('mean')

    # Constructor recent form
    df['constructor_recent_form'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5,min_periods=1).mean()
    )

    # Momentum feature
    df['driver_momentum'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].diff().fillna(0)

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# TIME SPLIT
# ============================================================

train_set = train[train['year'] <= 2018].copy()
valid_set = train[train['year'] > 2018].copy()

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

# ============================================================
# MODEL
# ============================================================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.02,

    num_leaves=63,

    feature_fraction=0.85,

    bagging_fraction=0.85,

    bagging_freq=5,

    n_estimators=7000,

    random_state=42,

    n_jobs=-1
)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(400),
        lgb.log_evaluation(200)

    ]
)

# VALIDATION

val_preds = model.predict(X_valid)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# TEST

test_preds = model.predict(
    test[features]
)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

# Race ranking

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# SUBMISSION

submission = pd.DataFrame({

    'id': test_ids,
    'finishing_position': test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.85, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.85
[LightGBM] [Warning] bagging_fraction is set=0.85, subsample=1.0 will be ignored. Current value: bagging_fraction=0.85
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.85, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.85
[LightGBM] [Warning] bagging_fraction is set=0.85, subsample=1.0 will be ignored. Current value: bagging_fraction=0.85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003323 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5018
[LightGBM] [Info] Number of data points in the train set: 24109, number of used features: 34
[Ligh

In [17]:
# ============================================================
# VERSION 10.1 — REFINED STRONG MODEL
# Focus: Improve Version 10 without breaking stability
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# LOAD DATA

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING (REFINED VERSION 10)
# ============================================================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['driverId','year','round'])

    # Fill missing qualifying

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    # Core features

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # Driver momentum

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    # Constructor strength (very useful)

    df['constructor_form'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    # Momentum shift

    df['momentum_delta'] = (
        df['driver_form_3'] -
        df['driver_form_5']
    )

    # Interaction features

    df['grid_x_form'] = (
        df['grid_rank'] *
        df['driver_form_3']
    )

    df['qual_x_form'] = (
        df['qual_rank'] *
        df['driver_form_3']
    )

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# TIME SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

# ============================================================
# LIGHTGBM MODEL (Better tuned)
# ============================================================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.015,

    num_leaves=72,

    max_depth=-1,

    feature_fraction=0.80,

    bagging_fraction=0.80,

    bagging_freq=5,

    min_child_samples=25,

    n_estimators=9000,

    random_state=42,

    n_jobs=-1
)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(500),
        lgb.log_evaluation(200)

    ]
)

# VALIDATION

val_preds = model.predict(X_valid)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# TEST

test_preds = model.predict(
    test[features]
)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

# Ranking fix

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# SUBMISSION

submission = pd.DataFrame({

    'id': test_ids,
    'finishing_position': test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003382 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5390
[LightGBM] [Info] Number of data points in the train set: 24109, number of used features: 35
[LightGBM] [W

In [18]:
# ============================================================
# VERSION 10.4 — FEATURE BOOST (BUILT ON VERSION 10)
# Focus: Add high-impact racing features
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# LOAD DATA

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['driverId','year','round'])

    # Fill missing qualifying

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    # Base features

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # =========================
    # DRIVER FORM FEATURES
    # =========================

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    # Driver consistency ⭐⭐⭐

    df['driver_std_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).std()
    )

    # Momentum

    df['momentum_delta'] = (
        df['driver_form_3']
        - df['driver_form_5']
    )

    # =========================
    # CONSTRUCTOR STRENGTH ⭐⭐⭐
    # =========================

    df['constructor_form_5'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    # =========================
    # GRID ADVANTAGE ⭐⭐
    # =========================

    df['grid_advantage'] = (
        df['grid_rank']
        - df['qual_rank']
    )

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# TIME SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

# ============================================================
# LIGHTGBM MODEL
# ============================================================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.015,

    num_leaves=72,

    feature_fraction=0.80,

    bagging_fraction=0.80,

    bagging_freq=5,

    min_child_samples=25,

    lambda_l1=0.3,

    lambda_l2=1.2,

    n_estimators=9000,

    random_state=42,

    n_jobs=-1
)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(500),
        lgb.log_evaluation(200)

    ]
)

# ============================================================
# VALIDATION
# ============================================================

val_preds = model.predict(X_valid)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# ============================================================
# TEST
# ============================================================

test_preds = model.predict(
    test[features]
)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# ============================================================
# SUBMISSION
# ============================================================

submission = pd.DataFrame({

    'id': test_ids,
    'finishing_position': test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[Ligh

In [19]:
# ============================================================
# VERSION 10.5 — INTERACTION BOOST
# Built on Version 10.4 (best performing base)
# Focus: Feature interactions (very high leaderboard impact)
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# LOAD DATA

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['driverId','year','round'])

    # Fill missing qualifying

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    # Base features

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # =========================
    # DRIVER FORM
    # =========================

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['driver_std_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).std()
    )

    df['momentum_delta'] = (
        df['driver_form_3']
        - df['driver_form_5']
    )

    # =========================
    # CONSTRUCTOR FORM
    # =========================

    df['constructor_form_5'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    # =========================
    # GRID ADVANTAGE
    # =========================

    df['grid_advantage'] = (
        df['grid_rank']
        - df['qual_rank']
    )

    # ============================================================
    # NEW — INTERACTION FEATURES ⭐⭐⭐
    # ============================================================

    # Driver performance × consistency

    df['form_consistency_interaction'] = (
        df['driver_form_5']
        * df['driver_std_5']
    )

    # Constructor strength × grid

    df['constructor_grid_interaction'] = (
        df['constructor_form_5']
        * df['grid_rank']
    )

    # Qualifying strength × momentum

    df['qual_momentum_interaction'] = (
        df['qual_gap']
        * df['momentum_delta']
    )

    # Driver form × constructor strength

    df['driver_constructor_strength'] = (
        df['driver_form_5']
        * df['constructor_form_5']
    )

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# TIME SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

# ============================================================
# MODEL — slightly tuned
# ============================================================

model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.014,

    num_leaves=75,

    feature_fraction=0.80,

    bagging_fraction=0.80,

    bagging_freq=5,

    min_child_samples=24,

    lambda_l1=0.3,

    lambda_l2=1.2,

    n_estimators=9500,

    random_state=42,

    n_jobs=-1
)

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(500),
        lgb.log_evaluation(200)

    ]
)

# ============================================================
# VALIDATION
# ============================================================

val_preds = model.predict(X_valid)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# ============================================================
# TEST
# ============================================================

test_preds = model.predict(
    test[features]
)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# ============================================================
# SUBMISSION
# ============================================================

submission = pd.DataFrame({

    'id': test_ids,
    'finishing_position': test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[Ligh

In [20]:
# ============================================================
# VERSION 10.6 — LIGHTGBM + CATBOOST ENSEMBLE
# Built on Version 10.4 best features
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# LOAD DATA

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING (Version 10.4 BEST)
# ============================================================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['driverId','year','round'])

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # Driver form

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['driver_std_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).std()
    )

    df['momentum_delta'] = (
        df['driver_form_3']
        - df['driver_form_5']
    )

    # Constructor strength

    df['constructor_form_5'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    # Grid advantage

    df['grid_advantage'] = (
        df['grid_rank']
        - df['qual_rank']
    )

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

# ============================================================
# MODEL 1 — LIGHTGBM
# ============================================================

lgb_model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.015,

    num_leaves=72,

    feature_fraction=0.80,

    bagging_fraction=0.80,

    bagging_freq=5,

    min_child_samples=25,

    lambda_l1=0.3,

    lambda_l2=1.2,

    n_estimators=9000,

    random_state=42,

    n_jobs=-1
)

lgb_model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(500),
        lgb.log_evaluation(200)

    ]
)

# ============================================================
# MODEL 2 — CATBOOST ⭐⭐⭐
# ============================================================

cat_model = CatBoostRegressor(

    iterations=6000,

    learning_rate=0.02,

    depth=6,

    loss_function='MAE',

    random_seed=42,

    verbose=500
)

cat_model.fit(

    X_train,
    y_train,

    eval_set=(X_valid,y_valid),

    early_stopping_rounds=500
)

# ============================================================
# VALIDATION ENSEMBLE
# ============================================================

lgb_val = lgb_model.predict(X_valid)
cat_val = cat_model.predict(X_valid)

val_preds = (

    0.6 * lgb_val +
    0.4 * cat_val

)

val_preds = np.clip(
    np.round(val_preds),
    1,
    20
)

val_mae = mean_absolute_error(
    y_valid,
    val_preds
)

print("Validation MAE:", val_mae)

# ============================================================
# TEST ENSEMBLE
# ============================================================

lgb_test = lgb_model.predict(test[features])
cat_test = cat_model.predict(test[features])

test_preds = (

    0.6 * lgb_test +
    0.4 * cat_test

)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# ============================================================
# SUBMISSION
# ============================================================

submission = pd.DataFrame({

    'id': test_ids,
    'finishing_position': test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[Ligh

In [21]:
# ============================================================
# VERSION 10.7 — ENSEMBLE WEIGHT OPTIMIZATION
# LightGBM + CatBoost with automatic weight search
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# LOAD DATA

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING (same as Version 10.4 / 10.6)
# ============================================================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['driverId','year','round'])

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['driver_std_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).std()
    )

    df['momentum_delta'] = (
        df['driver_form_3']
        - df['driver_form_5']
    )

    df['constructor_form_5'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['grid_advantage'] = (
        df['grid_rank']
        - df['qual_rank']
    )

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

# ============================================================
# MODEL 1 — LIGHTGBM
# ============================================================

lgb_model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.015,

    num_leaves=72,

    feature_fraction=0.80,

    bagging_fraction=0.80,

    bagging_freq=5,

    min_child_samples=25,

    lambda_l1=0.3,

    lambda_l2=1.2,

    n_estimators=9000,

    random_state=42,

    n_jobs=-1
)

lgb_model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(500),
        lgb.log_evaluation(200)

    ]
)

# ============================================================
# MODEL 2 — CATBOOST
# ============================================================

cat_model = CatBoostRegressor(

    iterations=6000,

    learning_rate=0.02,

    depth=6,

    loss_function='MAE',

    random_seed=42,

    verbose=500
)

cat_model.fit(

    X_train,
    y_train,

    eval_set=(X_valid,y_valid),

    early_stopping_rounds=500
)

# ============================================================
# VALIDATION PREDICTIONS
# ============================================================

lgb_val = lgb_model.predict(X_valid)
cat_val = cat_model.predict(X_valid)

# ============================================================
# FIND BEST WEIGHT ⭐
# ============================================================

best_weight = 0
best_score = 999

for w in np.arange(0.1, 0.9, 0.05):

    preds = (
        w * lgb_val +
        (1-w) * cat_val
    )

    preds = np.clip(
        np.round(preds),
        1,
        20
    )

    score = mean_absolute_error(
        y_valid,
        preds
    )

    if score < best_score:

        best_score = score
        best_weight = w

print("Best Weight:", best_weight)
print("Best Validation MAE:", best_score)

# ============================================================
# TEST USING BEST WEIGHT
# ============================================================

lgb_test = lgb_model.predict(test[features])
cat_test = cat_model.predict(test[features])

test_preds = (

    best_weight * lgb_test +
    (1-best_weight) * cat_test

)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# ============================================================
# SUBMISSION
# ============================================================

submission = pd.DataFrame({

    'id': test_ids,
    'finishing_position': test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[Ligh

In [22]:
# ============================================================
# VERSION 10.8 — TRIPLE MODEL ENSEMBLE
# LightGBM + CatBoost (depth 6) + CatBoost (depth 8)
# With automatic weight tuning
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# LOAD DATA

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING (same stable features)
# ============================================================

def feature_engineering(df):

    df = df.copy()

    df = df.sort_values(['driverId','year','round'])

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['grid_delta'] = df['grid'] - 1

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['driver_std_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).std()
    )

    df['momentum_delta'] = (
        df['driver_form_3']
        - df['driver_form_5']
    )

    df['constructor_form_5'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    df['grid_advantage'] = (
        df['grid_rank']
        - df['qual_rank']
    )

    df = df.fillna(-1)

    return df


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

# ============================================================
# MODEL 1 — LIGHTGBM
# ============================================================

lgb_model = lgb.LGBMRegressor(

    objective='regression_l1',

    learning_rate=0.015,

    num_leaves=72,

    feature_fraction=0.80,

    bagging_fraction=0.80,

    bagging_freq=5,

    min_child_samples=25,

    lambda_l1=0.3,

    lambda_l2=1.2,

    n_estimators=9000,

    random_state=42,

    n_jobs=-1
)

lgb_model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid,y_valid)],

    eval_metric='mae',

    callbacks=[

        lgb.early_stopping(500),
        lgb.log_evaluation(200)

    ]
)

# ============================================================
# MODEL 2 — CATBOOST depth=6
# ============================================================

cat_model_1 = CatBoostRegressor(

    iterations=6000,
    learning_rate=0.02,
    depth=6,
    loss_function='MAE',
    random_seed=42,
    verbose=500
)

cat_model_1.fit(

    X_train,
    y_train,

    eval_set=(X_valid,y_valid),
    early_stopping_rounds=500
)

# ============================================================
# MODEL 3 — CATBOOST depth=8 ⭐
# ============================================================

cat_model_2 = CatBoostRegressor(

    iterations=6000,
    learning_rate=0.02,
    depth=8,
    loss_function='MAE',
    random_seed=42,
    verbose=500
)

cat_model_2.fit(

    X_train,
    y_train,

    eval_set=(X_valid,y_valid),
    early_stopping_rounds=500
)

# ============================================================
# VALIDATION PREDICTIONS
# ============================================================

lgb_val = lgb_model.predict(X_valid)
cat1_val = cat_model_1.predict(X_valid)
cat2_val = cat_model_2.predict(X_valid)

# ============================================================
# SEARCH BEST WEIGHTS ⭐⭐⭐
# ============================================================

best_score = 999

for w1 in np.arange(0.1,0.6,0.1):

    for w2 in np.arange(0.1,0.6,0.1):

        w3 = 1 - w1 - w2

        if w3 <= 0:
            continue

        preds = (
            w1*lgb_val +
            w2*cat1_val +
            w3*cat2_val
        )

        preds = np.clip(
            np.round(preds),
            1,
            20
        )

        score = mean_absolute_error(
            y_valid,
            preds
        )

        if score < best_score:

            best_score = score
            best_weights = (w1,w2,w3)

print("Best Weights:", best_weights)
print("Best Validation MAE:", best_score)

# ============================================================
# TEST USING BEST WEIGHTS
# ============================================================

lgb_test = lgb_model.predict(test[features])
cat1_test = cat_model_1.predict(test[features])
cat2_test = cat_model_2.predict(test[features])

w1,w2,w3 = best_weights

test_preds = (
    w1*lgb_test +
    w2*cat1_test +
    w3*cat2_test
)

test_preds = np.clip(
    np.round(test_preds),
    1,
    20
)

test['finishing_position'] = test_preds

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

# SAVE

submission = pd.DataFrame({

    'id': test_ids,
    'finishing_position': test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[Ligh

In [23]:
# ============================================================
# VERSION 13 — FINAL OPTIMIZED ENSEMBLE
# LightGBM + CatBoost + Weight Search
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# ============================================================
# LOAD DATA
# ============================================================

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):

    df = df.sort_values(['driverId','year','round'])

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3,1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5,1).mean()
    )

    df['momentum_delta'] = (
        df['driver_form_3']
        - df['driver_form_5']
    )

    df['grid_advantage'] = (
        df['grid_rank']
        - df['qual_rank']
    )

    return df.fillna(-1)


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# TIME SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

X_test = test[features]

# ============================================================
# MODEL 1 — LIGHTGBM
# ============================================================

lgb_model = lgb.LGBMRegressor(

    objective='regression_l1',
    learning_rate=0.015,
    num_leaves=72,
    feature_fraction=0.80,
    bagging_fraction=0.80,
    bagging_freq=5,
    lambda_l1=0.3,
    lambda_l2=1.2,
    n_estimators=9000,
    random_state=42
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid,y_valid)],
    eval_metric='mae',
    callbacks=[
        lgb.early_stopping(300),
        lgb.log_evaluation(200)
    ]
)

pred_lgb_val = lgb_model.predict(X_valid)
pred_lgb_test = lgb_model.predict(X_test)

# ============================================================
# MODEL 2 — CATBOOST DEPTH 6
# ============================================================

cat1 = CatBoostRegressor(

    iterations=4000,
    learning_rate=0.02,
    depth=6,
    loss_function='MAE',
    random_seed=42,
    verbose=500
)

cat1.fit(
    X_train,
    y_train,
    eval_set=(X_valid,y_valid)
)

pred_cat1_val = cat1.predict(X_valid)
pred_cat1_test = cat1.predict(X_test)

# ============================================================
# MODEL 3 — CATBOOST DEPTH 8
# ============================================================

cat2 = CatBoostRegressor(

    iterations=4000,
    learning_rate=0.02,
    depth=8,
    loss_function='MAE',
    random_seed=42,
    verbose=500
)

cat2.fit(
    X_train,
    y_train,
    eval_set=(X_valid,y_valid)
)

pred_cat2_val = cat2.predict(X_valid)
pred_cat2_test = cat2.predict(X_test)

# ============================================================
# AUTOMATIC WEIGHT SEARCH
# ============================================================

best_mae = 999
best_weights = None

for w1 in np.arange(0.2,0.6,0.05):

    for w2 in np.arange(0.1,0.5,0.05):

        w3 = 1 - w1 - w2

        if w3 <= 0:
            continue

        val_pred = (
            w1 * pred_lgb_val +
            w2 * pred_cat1_val +
            w3 * pred_cat2_val
        )

        val_pred = np.clip(
            np.round(val_pred),
            1,
            20
        )

        mae = mean_absolute_error(
            y_valid,
            val_pred
        )

        if mae < best_mae:

            best_mae = mae
            best_weights = (w1,w2,w3)

print("Best Weights:",best_weights)
print("Best Validation MAE:",best_mae)

# ============================================================
# FINAL TEST PREDICTION
# ============================================================

w1,w2,w3 = best_weights

final_test = (
    w1 * pred_lgb_test +
    w2 * pred_cat1_test +
    w3 * pred_cat2_test
)

final_test = np.clip(
    np.round(final_test),
    1,
    20
)

test['finishing_position'] = final_test

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

submission = pd.DataFrame({

    'id':test_ids,
    'finishing_position':test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[Ligh

In [24]:
# ============================================================
# VERSION 15 — FEATURE INTERACTION BOOST
# (Based on your best working Version 13)
# ============================================================

import pandas as pd
import numpy as np

import lightgbm as lgb
from catboost import CatBoostRegressor

from sklearn.metrics import mean_absolute_error

# ============================================================
# LOAD DATA
# ============================================================

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING — STRONG BOOST
# ============================================================

def feature_engineering(df):

    df = df.sort_values(['driverId','year','round'])

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3,1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5,1).mean()
    )

    df['momentum_delta'] = (
        df['driver_form_3']
        - df['driver_form_5']
    )

    df['grid_advantage'] = (
        df['grid_rank']
        - df['qual_rank']
    )

    # NEW STRONG FEATURES

    df['qual_grid_product'] = (
        df['qual_rank']
        * df['grid_rank']
    )

    df['form_grid_product'] = (
        df['driver_form_5']
        * df['grid']
    )

    df['qual_gap_ratio'] = (
        df['qual_gap']
        / (df['grid'] + 1)
    )

    return df.fillna(-1)


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# TIME SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

X_test = test[features]

# ============================================================
# LIGHTGBM
# ============================================================

lgb_model = lgb.LGBMRegressor(

    objective='regression_l1',
    learning_rate=0.015,
    num_leaves=72,
    feature_fraction=0.80,
    bagging_fraction=0.80,
    bagging_freq=5,
    lambda_l1=0.3,
    lambda_l2=1.2,
    n_estimators=9000,
    random_state=42
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid,y_valid)],
    eval_metric='mae',
    callbacks=[
        lgb.early_stopping(300)
    ]
)

pred_lgb_val = lgb_model.predict(X_valid)
pred_lgb_test = lgb_model.predict(X_test)

# ============================================================
# CATBOOST MODELS
# ============================================================

cat1 = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.02,
    depth=6,
    loss_function='MAE',
    verbose=False
)

cat1.fit(
    X_train,
    y_train,
    eval_set=(X_valid,y_valid)
)

pred_cat1_val = cat1.predict(X_valid)
pred_cat1_test = cat1.predict(X_test)

cat2 = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.02,
    depth=8,
    loss_function='MAE',
    verbose=False
)

cat2.fit(
    X_train,
    y_train,
    eval_set=(X_valid,y_valid)
)

pred_cat2_val = cat2.predict(X_valid)
pred_cat2_test = cat2.predict(X_test)

# ============================================================
# WEIGHT SEARCH
# ============================================================

best_mae = 999
best_weights = None

for w1 in np.arange(0.2,0.6,0.05):

    for w2 in np.arange(0.1,0.5,0.05):

        w3 = 1 - w1 - w2

        if w3 <= 0:
            continue

        val_pred = (
            w1 * pred_lgb_val +
            w2 * pred_cat1_val +
            w3 * pred_cat2_val
        )

        val_pred = np.clip(
            np.round(val_pred),
            1,
            20
        )

        mae = mean_absolute_error(
            y_valid,
            val_pred
        )

        if mae < best_mae:

            best_mae = mae
            best_weights = (w1,w2,w3)

print("Best Weights:",best_weights)
print("Best Validation MAE:",best_mae)

# ============================================================
# FINAL TEST
# ============================================================

w1,w2,w3 = best_weights

final_test = (
    w1 * pred_lgb_test +
    w2 * pred_cat1_test +
    w3 * pred_cat2_test
)

final_test = np.clip(
    np.round(final_test),
    1,
    20
)

test['finishing_position'] = final_test

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

submission = pd.DataFrame({

    'id':test_ids,
    'finishing_position':test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[Ligh

In [25]:
# ============================================================
# VERSION 16 — DRIVER + CONSTRUCTOR STRENGTH BOOST
# Safe upgrade from Version 15
# ============================================================

import pandas as pd
import numpy as np

import lightgbm as lgb
from catboost import CatBoostRegressor

from sklearn.metrics import mean_absolute_error

# ============================================================
# LOAD DATA
# ============================================================

train = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv'
)

test = pd.read_csv(
'/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv'
)

test_ids = test['id']

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):

    df = df.sort_values(['driverId','year','round'])

    # -------------------------
    # Base Features
    # -------------------------

    df['best_qual_ms'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x.fillna(x.max())
    )

    df['qual_gap'] = df.groupby(
        'raceId'
    )['best_qual_ms'].transform(
        lambda x: x - x.min()
    )

    df['grid_norm'] = df.groupby(
        'raceId'
    )['grid'].transform(
        lambda x: x / x.max()
    )

    df['qual_rank'] = df.groupby(
        'raceId'
    )['best_qual_ms'].rank()

    df['grid_rank'] = df.groupby(
        'raceId'
    )['grid'].rank()

    # -------------------------
    # Driver Form
    # -------------------------

    df['driver_form_3'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(3,1).mean()
    )

    df['driver_form_5'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.rolling(5,1).mean()
    )

    df['momentum_delta'] = (
        df['driver_form_3']
        - df['driver_form_5']
    )

    # -------------------------
    # Driver Historical Skill
    # -------------------------

    df['driver_skill'] = df.groupby(
        'driverId'
    )['rolling_avg_position'].transform(
        lambda x: x.expanding().mean()
    )

    # -------------------------
    # Constructor Strength
    # -------------------------

    df['constructor_strength'] = df.groupby(
        'constructorId'
    )['rolling_avg_position'].transform(
        lambda x: x.expanding().mean()
    )

    # -------------------------
    # Interaction Features
    # -------------------------

    df['grid_advantage'] = (
        df['grid_rank']
        - df['qual_rank']
    )

    df['qual_grid_product'] = (
        df['qual_rank']
        * df['grid_rank']
    )

    df['form_grid_product'] = (
        df['driver_form_5']
        * df['grid']
    )

    df['qual_gap_ratio'] = (
        df['qual_gap']
        / (df['grid'] + 1)
    )

    df['driver_constructor_combo'] = (
        df['driver_skill']
        * df['constructor_strength']
    )

    return df.fillna(-1)


train = feature_engineering(train)
test = feature_engineering(test)

# ============================================================
# TIME SPLIT
# ============================================================

train_set = train[train['year'] <= 2018]
valid_set = train[train['year'] > 2018]

target = 'finishing_position'

drop_cols = [
    'id',
    'raceId',
    'finishing_position',
    'year'
]

features = [
    c for c in train.columns
    if c not in drop_cols
]

X_train = train_set[features]
y_train = train_set[target]

X_valid = valid_set[features]
y_valid = valid_set[target]

X_test = test[features]

# ============================================================
# LIGHTGBM MODEL
# ============================================================

lgb_model = lgb.LGBMRegressor(

    objective='regression_l1',
    learning_rate=0.015,
    num_leaves=80,
    feature_fraction=0.80,
    bagging_fraction=0.80,
    bagging_freq=5,
    lambda_l1=0.3,
    lambda_l2=1.2,
    n_estimators=10000,
    random_state=42
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid,y_valid)],
    eval_metric='mae',
    callbacks=[
        lgb.early_stopping(300)
    ]
)

pred_lgb_val = lgb_model.predict(X_valid)
pred_lgb_test = lgb_model.predict(X_test)

# ============================================================
# CATBOOST MODELS
# ============================================================

cat1 = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.02,
    depth=6,
    loss_function='MAE',
    verbose=False
)

cat1.fit(
    X_train,
    y_train,
    eval_set=(X_valid,y_valid)
)

pred_cat1_val = cat1.predict(X_valid)
pred_cat1_test = cat1.predict(X_test)

cat2 = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.02,
    depth=8,
    loss_function='MAE',
    verbose=False
)

cat2.fit(
    X_train,
    y_train,
    eval_set=(X_valid,y_valid)
)

pred_cat2_val = cat2.predict(X_valid)
pred_cat2_test = cat2.predict(X_test)

# ============================================================
# WEIGHT SEARCH
# ============================================================

best_mae = 999
best_weights = None

for w1 in np.arange(0.2,0.6,0.05):

    for w2 in np.arange(0.1,0.5,0.05):

        w3 = 1 - w1 - w2

        if w3 <= 0:
            continue

        val_pred = (
            w1 * pred_lgb_val +
            w2 * pred_cat1_val +
            w3 * pred_cat2_val
        )

        val_pred = np.clip(
            np.round(val_pred),
            1,
            20
        )

        mae = mean_absolute_error(
            y_valid,
            val_pred
        )

        if mae < best_mae:

            best_mae = mae
            best_weights = (w1,w2,w3)

print("Best Weights:",best_weights)
print("Best Validation MAE:",best_mae)

# ============================================================
# FINAL TEST
# ============================================================

w1,w2,w3 = best_weights

final_test = (
    w1 * pred_lgb_test +
    w2 * pred_cat1_test +
    w3 * pred_cat2_test
)

final_test = np.clip(
    np.round(final_test),
    1,
    20
)

test['finishing_position'] = final_test

test['finishing_position'] = test.groupby(
    'raceId'
)['finishing_position'].rank(
    method='first'
).astype(int)

submission = pd.DataFrame({

    'id':test_ids,
    'finishing_position':test['finishing_position']

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.2, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.2
[LightGBM] [Warning] lambda_l1 is set=0.3, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.3
[Ligh